In [1]:
import re
import pandas as pd
import unicodedata
import random

In [2]:
df = pd.read_csv('llm7.csv', encoding_errors='ignore')
df.head(2)

,text,label
0,Cars. Cars have been around since they became ...,0
1,Transportation is a large necessity in most co...,0


In [3]:
dfx = df.copy()
text = ' '.join(df.text.to_list())

In [4]:
# 1. Repair mojibake (turns \x91-\x99, \x80, \x85 back into real chars)
CP1252_FIXES = {
    "\x80": "€",  "\x85": "…",  "\x91": "'",  "\x92": "'",
    "\x93": '"',  "\x94": '"',  "\x97": "—",  "\x99": "™",
}
for bad, good in CP1252_FIXES.items():
    text = text.replace(bad, good)

In [5]:
# 2. Unicode-normalise: converts ² -> 2, full-width chars -> ASCII, etc.
text = unicodedata.normalize("NFKC", text)

In [6]:
# 3. Map typographic chars to ASCII equivalents
table = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"',
    "\u2013": "-", "\u2014": "-", "\u2012": "-", "\u2015": "-",
    "\u2026": "...", "\u00b4": "'", "\u00a8": '"',
    "\u00a0": " ",  # non-breaking space
    "\u2500": "-",  # box-drawing
})
text = text.translate(table)

In [7]:
# 4. Drop emoji, symbols, currency, marks (optional)
#    Categories: So = symbol-other, Sk = modifier, Sc = currency
text = "".join(c for c in text if unicodedata.category(c) not in {"So", "Sk", "Sc"})

In [8]:
# 5. Drop remaining control characters (Cc) except \n \t
text = "".join(
    c for c in text
    if unicodedata.category(c) != "Cc" or c in "\n\t"
)

In [9]:
# 6. Strip accents: á -> a, ö -> o
text = "".join(
    c for c in unicodedata.normalize("NFKD", text)
    if not unicodedata.combining(c)
)

In [10]:
# 7. Collapse whitespace
text = re.sub(r"\s+", " ", text).strip()
text = text.lower()

In [11]:
# 8. Optional: drop any remaining non-ASCII
# text = text.encode("ascii", "ignore").decode()

In [12]:
# exclude url > 
# exclude = #, &, <, >, ¶, þ

# % > ' percent '


# split = ("'")[0]

# *, left, right > replace, middle > exclude
# !" > space
# 0-9 > space
# (), space
# + > space
# , > space
# - > space
# . > space
# / > space
# : > space
# ? > space
# [] > space
# \\ > space
# _ > space
# {} > space
# | > space
# ~ > space

In [21]:
words = ' '.join(df.text.to_list()).lower().split()
words = list(dict.fromkeys(words))
len(words)

109821

In [22]:
# exclude
exc = ['#', '&', '<', '>', '¶', 'þ', 'http', 'www.']
words = [
    word for word in words
    if not any([ex in word for ex in exc])
]
len(words)

109678

In [23]:
# replace
words = [
    word.replace('%', ' percent ') for word in words
]
words = [
    word.replace('', ' percent ') for word in words
]